In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

Load Both Datasets

In [3]:
uci_df = pd.read_csv("../data/raw/ckd_dataset.csv")
kaggle_df = pd.read_csv("../data/raw/kaggle_ckd.csv")

print("UCI shape:", uci_df.shape)
print("Kaggle shape:", kaggle_df.shape)

UCI shape: (400, 25)
Kaggle shape: (400, 26)


Standardize Column Names

In [4]:
uci_df.columns = uci_df.columns.str.strip().str.lower().str.replace(' ', '_')
kaggle_df.columns = kaggle_df.columns.str.strip().str.lower().str.replace(' ', '_')

Identify & Rename Target Column

In [5]:
print(uci_df.columns)
print(kaggle_df.columns)

Index(['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu',
       'sc', 'sod', 'pot', 'hemo', 'pcv', 'wbcc', 'rbcc', 'htn', 'dm', 'cad',
       'appet', 'pe', 'ane', 'class'],
      dtype='object')
Index(['id', 'age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr',
       'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad',
       'appet', 'pe', 'ane', 'classification'],
      dtype='object')


In [8]:
uci_df.rename(columns={'class': 'classification'}, inplace=True)

In [9]:
print(uci_df.columns)
print(kaggle_df.columns)

Index(['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu',
       'sc', 'sod', 'pot', 'hemo', 'pcv', 'wbcc', 'rbcc', 'htn', 'dm', 'cad',
       'appet', 'pe', 'ane', 'classification'],
      dtype='object')
Index(['id', 'age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr',
       'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad',
       'appet', 'pe', 'ane', 'classification'],
      dtype='object')


Align Kaggle-Specific Columns

In [13]:
# Drop ID column if present
if 'id' in kaggle_df.columns:
    kaggle_df.drop(columns=['id'], inplace=True)

# Rename wc/rc to match UCI naming
kaggle_df.rename(columns={
    'wc': 'wbcc',
    'rc': 'rbcc'
}, inplace=True)

print("\nAfter alignment:")
print("UCI columns:\n", uci_df.columns)
print("Kaggle columns:\n", kaggle_df.columns)


After alignment:
UCI columns:
 Index(['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu',
       'sc', 'sod', 'pot', 'hemo', 'pcv', 'wbcc', 'rbcc', 'htn', 'dm', 'cad',
       'appet', 'pe', 'ane', 'classification'],
      dtype='object')
Kaggle columns:
 Index(['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu',
       'sc', 'sod', 'pot', 'hemo', 'pcv', 'wbcc', 'rbcc', 'htn', 'dm', 'cad',
       'appet', 'pe', 'ane', 'classification'],
      dtype='object')


Select COMMON FEATURES (Federated Requirement)

In [14]:
COMMON_FEATURES = [
    'age','bp','sg','al','su','rbc','pc','pcc','ba','bgr','bu',
    'sc','sod','pot','hemo','pcv','wbcc','rbcc','htn','dm','cad',
    'appet','pe','ane','classification'
]

uci_df = uci_df[COMMON_FEATURES]
kaggle_df = kaggle_df[COMMON_FEATURES]

print("\nFinal Shapes:")
print("UCI:", uci_df.shape)
print("Kaggle:", kaggle_df.shape)


Final Shapes:
UCI: (400, 25)
Kaggle: (400, 25)


Handle Missing Values

In [15]:
uci_df.replace(['?', '\t?'], np.nan, inplace=True)
kaggle_df.replace(['?', '\t?'], np.nan, inplace=True)


In [16]:
#Convert Numeric Columns Properly
num_cols = uci_df.columns.drop('classification')

for col in num_cols:
    uci_df[col] = pd.to_numeric(uci_df[col], errors='coerce')
    kaggle_df[col] = pd.to_numeric(kaggle_df[col], errors='coerce')

In [17]:
#Fill Missing Values
for col in num_cols:
    uci_df[col].fillna(uci_df[col].median(), inplace=True)
    kaggle_df[col].fillna(kaggle_df[col].median(), inplace=True)

d:\GitHub\FedGNN-CKD\fedgnn_env\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
d:\GitHub\FedGNN-CKD\fedgnn_env\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
d:\GitHub\FedGNN-CKD\fedgnn_env\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
d:\GitHub\FedGNN-CKD\fedgnn_env\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
d:\GitHub\FedGNN-CKD\fedgnn_env\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
d:\GitHub\FedGNN-CKD\fedgnn_env\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: Runtime

Encode Categorical Features

In [18]:
binary_map = {
    'yes': 1, 'no': 0,
    'ckd': 1, 'notckd': 0,
    'good': 1, 'poor': 0,
    'present': 1, 'notpresent': 0
}

uci_df.replace(binary_map, inplace=True)
kaggle_df.replace(binary_map, inplace=True)


C:\Users\GANESH\AppData\Local\Temp\ipykernel_14404\1140594843.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  uci_df.replace(binary_map, inplace=True)


Separate Features & Labels

In [19]:
X_uci = uci_df.drop('classification', axis=1)
y_uci = uci_df['classification']

X_kaggle = kaggle_df.drop('classification', axis=1)
y_kaggle = kaggle_df['classification']


Normalize Features (Federated Principle)

Each dataset is normalized independently.

In [20]:
scaler_uci = StandardScaler()
X_uci_scaled = scaler_uci.fit_transform(X_uci)

scaler_kaggle = StandardScaler()
X_kaggle_scaled = scaler_kaggle.fit_transform(X_kaggle)


d:\GitHub\FedGNN-CKD\fedgnn_env\Lib\site-packages\sklearn\utils\extmath.py:1207: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
d:\GitHub\FedGNN-CKD\fedgnn_env\Lib\site-packages\sklearn\utils\extmath.py:1212: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
d:\GitHub\FedGNN-CKD\fedgnn_env\Lib\site-packages\sklearn\utils\extmath.py:1236: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count
d:\GitHub\FedGNN-CKD\fedgnn_env\Lib\site-packages\sklearn\utils\extmath.py:1207: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
d:\GitHub\FedGNN-CKD\fedgnn_env\Lib\site-packages\sklearn\utils\extmath.py:1212: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
d:\GitHub\FedGNN-CKD\fedgnn_env\Lib\site-packages\sklearn\utils\extmath.py:1236: Ru

Save Processed Datasets

In [22]:
uci_clean = pd.DataFrame(X_uci_scaled, columns=X_uci.columns)
uci_clean['classification'] = y_uci.values
uci_clean.to_csv("../data/processed/uci_clean.csv", index=False)

kaggle_clean = pd.DataFrame(X_kaggle_scaled, columns=X_kaggle.columns)
kaggle_clean['classification'] = y_kaggle.values
kaggle_clean.to_csv("../data/processed/kaggle_clean.csv", index=False)


In [24]:
uci_clean.head()

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane,classification
0,-0.210031,0.254214,0.421486,0.076249,-0.380269,NaN,NaN,NaN,NaN,-0.320122,...,0.603224,-0.197314,0.550044,NaN,NaN,NaN,NaN,NaN,NaN,1
1,-2.627234,-1.972476,0.421486,2.363728,-0.380269,NaN,NaN,NaN,NaN,-0.320122,...,-0.132789,-0.909782,0.074073,NaN,NaN,NaN,NaN,NaN,NaN,1
2,0.615355,0.254214,-1.421074,0.838742,2.507853,NaN,NaN,NaN,NaN,3.697618,...,-0.991470,-0.316059,0.074073,NaN,NaN,NaN,NaN,NaN,NaN,1
3,-0.210031,-0.488016,-2.342354,2.363728,-0.380269,NaN,NaN,NaN,NaN,-0.373337,...,-0.868801,-0.632711,-0.996862,NaN,NaN,NaN,NaN,NaN,NaN,1
4,-0.033163,0.254214,-1.421074,0.838742,-0.380269,NaN,NaN,NaN,NaN,-0.519679,...,-0.500795,-0.395222,-0.163913,NaN,NaN,NaN,NaN,NaN,NaN,1


In [25]:
kaggle_clean.head()

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane,classification
0,-0.210031,0.254214,0.421486,0.076249,-0.380269,NaN,NaN,NaN,NaN,-0.320122,...,0.603224,-0.197314,0.550044,NaN,NaN,NaN,NaN,NaN,NaN,1
1,-2.627234,-1.972476,0.421486,2.363728,-0.380269,NaN,NaN,NaN,NaN,-0.320122,...,-0.132789,-0.909782,0.074073,NaN,NaN,NaN,NaN,NaN,NaN,1
2,0.615355,0.254214,-1.421074,0.838742,2.507853,NaN,NaN,NaN,NaN,3.697618,...,-0.991470,-0.316059,0.074073,NaN,NaN,NaN,NaN,NaN,NaN,1
3,-0.210031,-0.488016,-2.342354,2.363728,-0.380269,NaN,NaN,NaN,NaN,-0.373337,...,-0.868801,-0.632711,-0.996862,NaN,NaN,NaN,NaN,NaN,NaN,1
4,-0.033163,0.254214,-1.421074,0.838742,-0.380269,NaN,NaN,NaN,NaN,-0.519679,...,-0.500795,-0.395222,-0.163913,NaN,NaN,NaN,NaN,NaN,NaN,1
